data source: [Healthy China](https://weekly.chinacdc.cn/en/article/doi/10.46234/ccdcw2023.008)<br>
[List of Chinese administrative divisions by life expectancy](https://en.wikipedia.org/wiki/List_of_Chinese_administrative_divisions_by_life_expectancy) / [Продолжительность жизни в регионах Китая](https://ru.wikipedia.org/wiki/Продолжительность_жизни_в_регионах_Китая)<br>
[MapChart](https://www.mapchart.net/china.html)

In [2]:
import pandas as pd
import math
import re
from collections import namedtuple

In [3]:
df = pd.read_csv('data/China_2019.csv', sep='\t', index_col='region')
df.index.name = ''

df.head(3)

,1990_total,1990_male,1990_female,2019_total,2019_male,2019_female,Δ_total,Δ_male,Δ_female
,,,,,,,,,
China,68.0,66.1,70.2,77.6,74.7,80.7,9.6,8.6,10.5
Beijing,73.6,71.4,76.2,82.7,79.6,85.9,9.1,8.2,9.7
Shanghai,74.5,72.4,76.6,82.7,80.3,85.0,8.2,7.9,8.4


In [4]:
df = df.loc[:, ['2019_total', '2019_male', '2019_female']] \
       .rename(columns = {'2019_total' : 'total',
                          '2019_male' : 'male',
                          '2019_female' : 'female'})

df.insert(loc=3, column='fΔm',  value=(df['female']-df['male']).round(1))

df.sort_values(by=['total', 'male'], ascending=False, inplace=True)
df = pd.concat([df.loc[['China']], df.drop('China')])

df

,total,male,female,fΔm
,,,,
China,77.6,74.7,80.7,6.0
Shanghai,82.7,80.3,85.0,4.7
Beijing,82.7,79.6,85.9,6.3
Zhejiang,81.5,78.8,84.5,5.7
Guangdong,81.1,77.8,84.7,6.9
Jiangsu,80.8,77.9,83.8,5.9
Fujian,79.7,76.6,83.1,6.5
Tianjin,79.5,76.8,82.7,5.9
Shandong,78.7,75.7,81.9,6.2


<br>
<br>

In [6]:
# Some inner statistics for interest:
def min_and_max(df, region_center=''):
    print(f"Number of values: {len(df)}")
    
    def find_3_largest(df):
        t_df = df.nlargest(3)
        t_df = t_df.reset_index()
        return t_df.apply(lambda v: f"{v.iloc[1]:.1f} – {v.iloc[0]}", axis=1)

    def find_3_smallest(df):
        t_df = df.nsmallest(3).iloc[::-1]
        t_df = t_df.reset_index()
        return t_df.apply(lambda v: f"{v.iloc[1]:.1f} – {v.iloc[0]}", axis=1)
    
    if region_center:
        ser_center = df.loc[region_center].map(lambda v: f" — {v} —")
        final_df = pd.concat([df.apply(find_3_largest), ser_center.to_frame().T, df.apply(find_3_smallest)])
        final_df.index = ['max', 'max_2', 'max_3', region_center, 'min_3', 'min_2', 'min']
    else:
        final_df = pd.concat([df.apply(find_3_largest), df.apply(find_3_smallest)])
        final_df.index = ['max', 'max_2', 'max_3', 'min_3', 'min_2', 'min']
    return final_df.style.set_properties(**{'text-align': 'left'})

min_and_max(df, region_center="China")

Number of values: 32


,total,male,female,fΔm
max,82.7 – Shanghai,80.3 – Shanghai,85.9 – Beijing,7.6 – Guangxi
max_2,82.7 – Beijing,79.6 – Beijing,85.0 – Shanghai,7.2 – Liaoning
max_3,81.5 – Zhejiang,78.8 – Zhejiang,84.7 – Guangdong,6.9 – Guangdong
China,— 77.6 —,— 74.7 —,— 80.7 —,— 6.0 —
min_3,72.1 – Qinghai,70.1 – Qinghai,74.4 – Qinghai,4.3 – Qinghai
min_2,71.9 – Xinjiang,70.0 – Xinjiang,74.2 – Xinjiang,4.2 – Xinjiang
min,70.1 – Tibet,67.6 – Tibet,72.9 – Tibet,2.6 – Jilin


<br>
<br>

In [8]:
# for region in sorted(df.index.to_list()):
#     print(f"    '{region}': {{'en': ('', ''), 'ru': ('', '')}},")

In [9]:
dd_replacement = {
    'China': {'en': ('China', ''), 'ru': ('Китай в среднем', '')},
    'Anhui': {'en': ('Anhui', 'Anhui'), 'ru': ('Аньхо́й', 'Аньхой')},
    'Beijing': {'en': ('Beijing', 'Beijing'), 'ru': ('Пеки́н', 'Пекин')},
    'Chongqing': {'en': ('Chongqing', 'Chongqing'), 'ru': ('Чунци́н', 'Чунцин')},
    'Fujian': {'en': ('Fujian', 'Fujian'), 'ru': ('Фуцзя́нь', 'Фуцзянь')},
    'Gansu': {'en': ('Gansu', 'Gansu'), 'ru': ('Ганьсу́', 'Ганьсу')},
    'Guangdong': {'en': ('Guangdong', 'Guangdong'), 'ru': ('Гуанду́н', 'Гуандун')},
    'Guangxi': {'en': ('Guangxi', 'Guangxi'), 'ru': ('Гуанси́-Чжуа́нский АО', 'Гуанси-Чжуанский автономный район')},
    'Guizhou': {'en': ('Guizhou', 'Guizhou'), 'ru': ('Гуйчжо́у', 'Гуйчжоу')},
    'Hainan': {'en': ('Hainan', 'Hainan'), 'ru': ('Хайна́нь', 'Хайнань')},
    'Hebei': {'en': ('Hebei', 'Hebei'), 'ru': ('Хэбэ́й', 'Хэбэй')},
    'Heilongjiang': {'en': ('Heilongjiang', 'Heilongjiang'), 'ru': ('Хэйлунцзя́н', 'Хэйлунцзян')},
    'Henan': {'en': ('Henan', 'Henan'), 'ru': ('Хэна́нь', 'Хэнань')},
    'Hubei': {'en': ('Hubei', 'Hubei'), 'ru': ('Хубэ́й', 'Хубэй')},
    'Hunan': {'en': ('Hunan', 'Hunan'), 'ru': ('Хуна́нь', 'Хунань')},
    'Inner Mongolia': {'en': ('Inner Mongolia', 'Inner Mongolia'), 'ru': ('Внутренняя Монголия', 'Внутренняя Монголия')},
    'Jiangsu': {'en': ('Jiangsu', 'Jiangsu'), 'ru': ('Цзянсу́', 'Цзянсу')},
    'Jiangxi': {'en': ('Jiangxi', 'Jiangxi'), 'ru': ('Цзянси́', 'Цзянси')},
    'Jilin': {'en': ('Jilin', 'Jilin'), 'ru': ('Гири́н', 'Гирин')},
    'Liaoning': {'en': ('Liaoning', 'Liaoning'), 'ru': ('Ляонин', 'Ляонин')},
    'Ningxia': {'en': ('Ningxia', 'Ningxia'), 'ru': ('Нинся́-Хуэйский АО', 'Нинся-Хуэйский автономный район')},
    'Qinghai': {'en': ('Qinghai', 'Qinghai'), 'ru': ('Цинха́й', 'Цинхай')},
    'Shaanxi': {'en': ('Shaanxi', 'Shaanxi'), 'ru': ('Шэньси́', 'Шэньси')},
    'Shandong': {'en': ('Shandong', 'Shandong'), 'ru': ('Шаньду́н', 'Шаньдун')},
    'Shanghai': {'en': ('Shanghai', 'Shanghai'), 'ru': ('Шанха́й', 'Шанхай')},
    'Shanxi': {'en': ('Shanxi', 'Shanxi'), 'ru': ('Шаньси́', 'Шаньси')},
    'Sichuan': {'en': ('Sichuan', 'Sichuan'), 'ru': ('Сычуа́нь', 'Сычуань')},
    'Tianjin': {'en': ('Tianjin', 'Tianjin'), 'ru': ('Тяньцзи́нь', 'Тяньцзинь')},
    'Tibet': {'en': ('Tibet', 'Tibet Autonomous Region'), 'ru': ('Тибетский АР', 'Тибетский автономный район')},
    'Xinjiang': {'en': ('Xinjiang', 'Xinjiang'), 'ru': ('Синьцзя́н-Уйгу́рский АР', 'Синьцзян-Уйгурский автономный район')},
    'Yunnan': {'en': ('Yunnan', 'Yunnan'), 'ru': ('Юньна́нь', 'Юньнань')},
    'Zhejiang': {'en': ('Zhejiang', 'Zhejiang'), 'ru': ('Чжэцзя́н', 'Чжэцзян')}
}

In [10]:
# create code for placing info in Wikipedia
def create_table(df, file_header, lang='ru'):

    def if_value(x, prec=1):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)

    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]
        if ser.name in ['China']:
             st += '\n' + '|-class=static-row-header\n' + \
                  f'|style="text-align:left;"|\'\'\'{dd_replacement[ser.name][lang][0]}\'\'\' ' + \
                  f'|| style="text-align:center;background:#e0ffd8;"|\'\'\'{if_value(ser["total"])}\'\'\' ' + \
                  f'|| style="text-align:center;background:#eaf3ff;"|\'\'\'{if_value(ser["male"])}\'\'\' ' + \
                  f'|| style="text-align:center;background:#fee7f6;"|\'\'\'{if_value(ser["female"])}\'\'\' ' + \
                  f'|| style="text-align:center;"|\'\'\'{if_value(ser["fΔm"])}\'\'\''
        else:
            name_link = dd_replacement[ser.name][lang][1]
            name_visible = dd_replacement[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'|style="text-align:left;"|[[{name_inserted}]] ' + \
                  f'|| style="text-align:center;background:#e0ffd8;"|{if_value(ser["total"])} ' + \
                  f'|| style="text-align:center;background:#eaf3ff;"|{if_value(ser["male"])} ' + \
                  f'|| style="text-align:center;background:#fee7f6;"|{if_value(ser["female"])} ' + \
                  f'|| style="text-align:center;"|{if_value(ser["fΔm"])}'

    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"|—', ';color:silver;"|—')

    return st


table_code = create_table(df, file_header='China_header_ru -2019.txt', lang='ru')

# write the code to file
with open('output/Table code for Chinese regions -ru.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

In [11]:
table_code = create_table(df, file_header='China_header_en -2019.txt', lang='en')

# write the code to file
with open('output/Table code for Chinese regions -en.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

<br>
<br>
<br>
<hr>

<h3>Map creation</h3>

In [13]:
CountryGroup = namedtuple('CountryGroup', ['group_label', 'color', 'countries'])

In [14]:
# for state in sorted(df.index.to_list()):
#     print(f"'{state}'", end=', ')

In [15]:
df_map = df.copy()                                    \
           .drop(['China']) \
           .rename(index={
               'Tibet' : 'Tibet Autonomous Region'
           })

df_map.head(3).fillna('')

,total,male,female,fΔm
,,,,
Shanghai,82.7,80.3,85.0,4.7
Beijing,82.7,79.6,85.9,6.3
Zhejiang,81.5,78.8,84.5,5.7


In [16]:
dd_legend = {
    # '85.0–85.9' : '002000',
    # '84.0–84.9' : '006000',
    # '83.0–83.9' : '009000',
    '82.0–82.9' : '00b800',
    '81.0–81.9' : '00e000',
    '80.0–80.9' : '00ff00',
    '79.0–79.9' : 'b8ff00',
    '78.0–78.9' : 'ffff00',
    '77.0–77.9' : 'ffe000',
    '76.0–76.9' : 'ffc000',
    '75.0–75.9' : 'ffa000',
    '74.0–74.9' : 'ff8000',
    '73.0–73.9' : 'ff5000',
    '72.0–72.9' : 'ff0000',
    '71.0–71.9' : 'c80000',
    '70.0–70.9' : '900000'
}

def create_legend_code(dd_legend):
    for k, v in dd_legend.items():
        print(f"{{{{Legend|#{v}|{k}}}}}")

create_legend_code(dd_legend)

{{Legend|#00b800|82.0–82.9}}
{{Legend|#00e000|81.0–81.9}}
{{Legend|#00ff00|80.0–80.9}}
{{Legend|#b8ff00|79.0–79.9}}
{{Legend|#ffff00|78.0–78.9}}
{{Legend|#ffe000|77.0–77.9}}
{{Legend|#ffc000|76.0–76.9}}
{{Legend|#ffa000|75.0–75.9}}
{{Legend|#ff8000|74.0–74.9}}
{{Legend|#ff5000|73.0–73.9}}
{{Legend|#ff0000|72.0–72.9}}
{{Legend|#c80000|71.0–71.9}}
{{Legend|#900000|70.0–70.9}}


In [17]:
def filter_df(df, selected_column):
    filtered_df = df.loc[:, [selected_column]]   \
                    .sort_values(by=selected_column, ascending=False) \
                    .dropna()

    filtered_df['group_label'] = filtered_df[selected_column].map(lambda x: f"{int(x):.1f}–{int(x) + 0.9:.1f}")
    
    
    # filtered_df['group_label'] = filtered_df['group_label'].replace(['79.5–79.99', '79.0–79.49', '78.5–78.99', '78.0–78.49'], '78.0–79.99')
    
    min_value = filtered_df[selected_column].min()
    max_value = filtered_df[selected_column].max()
    
    print(f"           ——— {selected_column} ———")
    print(f"Range: {min_value:.1f} – {max_value:.1f}   " +
          f"({filtered_df[selected_column].idxmin()} – {filtered_df[selected_column].idxmax()})")
    print(f"Number of groups: {filtered_df['group_label'].nunique()}")
    print(f"Number of values: {len(filtered_df)}")

    return filtered_df


df_sel = filter_df(df_map, selected_column='total')
df_sel

           ——— total ———
Range: 70.1 – 82.7   (Tibet Autonomous Region – Shanghai)
Number of groups: 12
Number of values: 31


,total,group_label
,,
Shanghai,82.7,82.0–82.9
Beijing,82.7,82.0–82.9
Zhejiang,81.5,81.0–81.9
Guangdong,81.1,81.0–81.9
Jiangsu,80.8,80.0–80.9
Fujian,79.7,79.0–79.9
Tianjin,79.5,79.0–79.9
Shandong,78.7,78.0–78.9
Anhui,78.0,78.0–78.9


In [18]:
def extract_indexes(subdf, dd_legend = dd_legend):
    group_label = subdf['group_label'].iloc[0]
    countries = subdf.index.to_list()
    color = (dd_legend[group_label])
    
    ls_grouping.append(CountryGroup(group_label=group_label, countries=countries, color=color))

    return pd.Series([color, countries], index=['color', 'regions'])


ls_grouping = []
df_grouped = df_sel.groupby(['group_label'])[['group_label']].apply(extract_indexes).loc[::-1]

df_grouped

,color,regions
group_label,,
82.0–82.9,00b800,"[Shanghai, Beijing]"
81.0–81.9,00e000,"[Zhejiang, Guangdong]"
80.0–80.9,00ff00,[Jiangsu]
79.0–79.9,b8ff00,"[Fujian, Tianjin]"
78.0–78.9,ffff00,"[Shandong, Anhui]"
77.0–77.9,ffe000,"[Jiangxi, Hubei, Henan, Liaoning, Jilin, Hainan]"
76.0–76.9,ffc000,"[Shaanxi, Hunan, Chongqing, Shanxi, Inner Mong..."
75.0–75.9,ffa000,"[Ningxia, Hebei, Heilongjiang, Sichuan]"
74.0–74.9,ff8000,"[Guizhou, Yunnan]"


In [19]:
# add record for n/a states
states_na = ['Hong Kong', 'Macau']

ls_on_map = df_map.index.to_list()

assert not [state for state in states_na if state in ls_on_map], "some states, designed to be assign as N/A, are really have values"

ls_grouping.insert(0, CountryGroup(group_label='n/a', countries=states_na, color='e0e0e0'))

In [20]:
def create_map_code_regions(ls_grouping, title=''):
    st = '{"groups":{'
    for group_label, color, regions in ls_grouping[::-1]:
        st_ls_regions = '"' + '","'.join(regions) + '"'
        st_ls_regions = st_ls_regions.replace(' ', '_')
        st += f'"#{color}":{{"label":"{group_label}","paths":[{st_ls_regions}]}},'

    st = st[:-1] + '},"title":"' + title + \
         '","hidden":[],"background":"#fff","borders":"#000","legendFont":"Century Gothic","legendFontColor":"#000","legendBgColor":"#00000000","legendBoxShape":"square","legendBorderColor":"#00000000","legendWidth":459.4714285714289,"areBordersShown":true,"defaultColor":"#d1dbdd","labelsColor":"#6a0707","labelsFont":"Arial","strokeWidth":"medium","areLabelsShown":true,"uncoloredScriptColor":"#ffff33","v5":true,"legendPosition":"custom","legendX":29.09999999999978,"legendY":988.9590921932383,"canvasWidth":1818,"canvasHeight":1378,"legendSize":"large","legendStatus":"show","scalingPatterns":true,"legendRowsSameColor":true,"legendColumnCount":2}'
    
    return st


st = create_map_code_regions(ls_grouping, title='2019')
print(st)

{"groups":{"#00b800":{"label":"82.0–82.9","paths":["Shanghai","Beijing"]},"#00e000":{"label":"81.0–81.9","paths":["Zhejiang","Guangdong"]},"#00ff00":{"label":"80.0–80.9","paths":["Jiangsu"]},"#b8ff00":{"label":"79.0–79.9","paths":["Fujian","Tianjin"]},"#ffff00":{"label":"78.0–78.9","paths":["Shandong","Anhui"]},"#ffe000":{"label":"77.0–77.9","paths":["Jiangxi","Hubei","Henan","Liaoning","Jilin","Hainan"]},"#ffc000":{"label":"76.0–76.9","paths":["Shaanxi","Hunan","Chongqing","Shanxi","Inner_Mongolia","Gansu","Guangxi"]},"#ffa000":{"label":"75.0–75.9","paths":["Ningxia","Hebei","Heilongjiang","Sichuan"]},"#ff8000":{"label":"74.0–74.9","paths":["Guizhou","Yunnan"]},"#ff0000":{"label":"72.0–72.9","paths":["Qinghai"]},"#c80000":{"label":"71.0–71.9","paths":["Xinjiang"]},"#900000":{"label":"70.0–70.9","paths":["Tibet_Autonomous_Region"]},"#e0e0e0":{"label":"n/a","paths":["Hong_Kong","Macau"]}},"title":"2019","hidden":[],"background":"#fff","borders":"#000","legendFont":"Century Gothic","lege